# 02 · Interpretable lexical retrieval

**Goal:** compare weighted token overlap with BM25, inspect hard metadata filters, and observe vocabulary limitations before adding embeddings. Both algorithms live in the Python package.

## Step 1 — Load and tokenize

In [ ]:
from savoroute.data.catalog import load_catalog
from savoroute.retrieval.lexical import (
    SearchFilters, filter_catalog, search_catalog, tokenize
)

items = load_catalog().items
print("Tokens:", tokenize("Cozy vegetarian noodles!"))
print("Catalog size:", len(items))

## Step 2 — Compare ranking functions

Overlap counts shared terms with fixed field weights. BM25 also accounts for term frequency, document length and inverse document frequency. The absolute scores are **not** comparable between methods.

In [ ]:
query = "spicy tofu noodles"
for method in ("overlap", "bm25"):
    print("\nMethod:", method)
    for hit in search_catalog(items, query, method=method, limit=4):
        print(hit.item.item_id, hit.item.title, round(hit.score, 3), hit.matched_terms)

## Step 3 — Apply metadata constraints as hard filters

Only matching catalog metadata qualifies. A restaurant tagged `vegan-options` is not automatically tagged `vegan`.

In [ ]:
rules = SearchFilters(
    kind="restaurant", cuisine="Mediterranean", location="Sample City",
    max_price_tier=2, dietary_tags=("vegan-options",)
)
selected = filter_catalog(items, rules)
results = search_catalog(items, "hummus vegetables", rules, method="bm25")
print("Eligible IDs:", [item.item_id for item in selected])
print("Ranked IDs:", [hit.item.item_id for hit in results])
assert [hit.item.item_id for hit in results] == ["r002"]

## Step 4 — Examine BM25 score stability

Global catalog statistics are used even if we filter the candidate records. Scores are *not* a calibrated relevance probability.

In [ ]:
full = search_catalog(items, "pasta", method="bm25", limit=20)
filtered = search_catalog(items, "pasta", SearchFilters(kind="recipe"),
                          method="bm25", limit=20)
lookup = {hit.item.item_id: hit.score for hit in full}
for hit in filtered:
    assert lookup[hit.item.item_id] == hit.score
print("Scores preserved under filtering:", True)

## Step 5 — Test a paraphrase

The current tokenizer does not resolve synonyms. A dense-retrieval baseline is part of the subsequent indexing stage.

In [ ]:
query = "legume wraps"
for method in ("overlap", "bm25"):
    hits = search_catalog(items, query, SearchFilters(kind="recipe"), method=method)
    print(method, [(hit.item.item_id, hit.item.title) for hit in hits])
print("Intended relevant item in the synthetic judgment set: p006")

Next: [`03_retrieval_evaluation.ipynb`](03_retrieval_evaluation.ipynb).